In [1]:
from pathlib import Path

import torch
from transformers import AutoTokenizer, LlamaForCausalLM, TextStreamer

In [2]:
data_dir = Path("data") / "ch12"
data_dir.mkdir(parents=True, exist_ok=True)

ckpt_dir = data_dir / "checkpoint-500" # "best"

In [3]:
# 推荐：left padding + 用 eos 充当 pad（纯生成最省事）
tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True, local_files_only=True)
tokenizer.padding_side = "left"

#streamer = TextStreamer(tokenizer)

model = LlamaForCausalLM.from_pretrained(
    ckpt_dir,
    local_files_only=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",           # 多卡自动放置
    attn_implementation="sdpa",  # 若已安装 flash-attn, 则使用 flash_attention_2；否则删掉或改 "sdpa"
)

model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_disable()
model.config.use_cache = True

In [5]:
#msg = "Tell me a joke"
msg = "Waht's ChatGPT?"

inputs = tokenizer(
    msg, return_tensors="pt",
    add_special_tokens=False,
    return_attention_mask=True,
    return_token_type_ids=False,
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    #streamer=streamer,
    use_cache=True,
    #pad_token_id=tokenizer.pad_token_id,
)

answer = tokenizer.decode(list(outputs[0]), skip_special_tokens=True)
print(f"answer: {answer}")

answer: Waht's ChatGPT?It for a best is away?I people really very clearing in?The most most "Sureot?Which, the people in some unique question or.The ""The Nle to Ath?A men was much to explain the game for human it is a different for the form and other to have the solving of a new.In many questions?Ok. I! The software in Earth about I dos the Pet that would be be unlike m.I'm about my work!The La?The Sents?I am my story?Thank you. I believ
